# 에이전트를 시나리오로 자동 시험하고 채점하기

새한님, **`USE = "fake"`로 위에서부터 실행하고 Solar 셀은 건너뛰세요.** 같은 고객 대본으로 두 상담 에이전트를 시험합니다. 도구와 주문 상태를 검사한 뒤, 완료 안내까지 검사하면 점수가 어떻게 달라지는지 봅니다. 주문은 모두 실습용이에요.

| 새 용어 | 여기서 뜻하는 것 |
|---|---|
| 시나리오 | 고객 대본과 통과 조건을 함께 담은 시험 사례예요. |
| 실행기(runner) | 대본을 한 줄씩 보내고 실행 중에 생긴 일을 모아요. |
| 실행 궤적(trajectory) | 고객 발화부터 도구 요청, 실행, 답변까지 턴별로 남긴 기록이에요. |
| 채점기(grader) | 정해 둔 검사 규칙을 실행 기록에 적용해요. |
| 허위 완료(false completion) | 작업이 성공했다는 답변을 뒷받침할 실행 결과가 없는 경우예요. |
| 통과율 | 전체 실행 중 모든 검사에 통과한 실행의 비율이에요. |

HITL과 기억의 기본은 [대화 기억과 사람의 승인](hitl_memory_playground.ipynb)에서 다시 볼 수 있어요. 여러 턴에 걸친 주문 참조는 [반품 상담](scenario_return_multiturn.ipynb), 담당 전환은 [담당 인계](scenario_specialist_handoff.ipynb), 도구 실패와 재생은 [오류와 체크포인트 분기](scenario_tool_error_time_travel.ipynb), 긴 대화와 묶음 승인은 [긴 상담의 기억](scenario_long_session_memory.ipynb)으로 돌아가면 됩니다.

## 준비

프로젝트 경로를 맞추고 지금 커널의 Python과 패키지 버전을 출력합니다. `show()`는 메시지를 한 줄씩 보여 줍니다. 모델의 도구 요청에는 이름과 인자를, 도구 응답에는 상태와 내용을 붙여요. 모델의 도구 요청을 읽을 수 없으면 그 건수를 보여 줍니다. `thread_config()`는 시나리오를 한 번 돌릴 때마다 새 대화 이름표를 만듭니다.

In [ ]:
import json
import os
import re
import subprocess
import sys
from collections import Counter
from copy import deepcopy
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

sys.dont_write_bytecode = True

REPO = Path("/home/home/projects/shopagent")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import ToolRuntime, tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

print(f"Python {sys.version.split()[0]}")
for package in ("langchain", "langchain-core", "langgraph", "langchain-openai", "ipykernel"):
    print(f"{package} {version(package)}")


def call_text(call):
    return f"{call['name']}({json.dumps(call['args'], ensure_ascii=False)})"


def show(messages):
    labels = {"human": "고객", "ai": "모델", "tool": "도구 응답"}
    for message in messages:
        line = str(message.text).replace("\n", " ↵ ").strip()
        if isinstance(message, AIMessage) and message.tool_calls:
            calls = ", ".join(call_text(call) for call in message.tool_calls)
            line = f"{line} | 요청 {calls}" if line else f"요청 {calls}"
        if isinstance(message, AIMessage) and message.invalid_tool_calls:
            invalid = f"형식을 읽을 수 없는 요청 {len(message.invalid_tool_calls)}건"
            line = f"{line} | {invalid}" if line else invalid
        if isinstance(message, ToolMessage):
            line = f"{message.name} [{message.status}] {line}"
        if line:
            print(f"  {labels.get(message.type, message.type)} · {line}")


def thread_config(label):
    return {"configurable": {"thread_id": f"{label}-{uuid4().hex[:8]}"}}

### Gemma 연결 준비

`LocalLLM(port=18380).client`로 연결 객체를 준비합니다. `/health`의 HTTP 상태 코드가 2xx면 Gemma를 선택할 수 있어요. 응답이 없으면 안내를 출력하고 다음 셀로 넘어갑니다.

나중에 Gemma로 실습하려면 별도 터미널에서 다음 명령을 실행하세요. 종료할 때는 그 터미널에서 Ctrl+C를 누르세요. 가짜 모델에는 서버가 필요 없어요.

~~~bash
/home/home/llama.cpp/build/bin/llama-server -m /home/home/models/gemma-4-12b-it-Q4_K_M.gguf --jinja -ngl 99 -ctk q8_0 -ctv q8_0 -fa on -np 1 -c 16384 --host 127.0.0.1 --port 18380 --n-cpu-moe 0 --chat-template-kwargs '{"enable_thinking": false}' --reasoning-budget 0
~~~

In [ ]:
import httpx
from shop.agents import LocalLLM

gemma_llm = LocalLLM(port=18380).client
gemma_ready = False
try:
    with httpx.Client(trust_env=False) as health_client:
        response = health_client.get("http://127.0.0.1:18380/health", timeout=2)
    gemma_ready = response.is_success
    print(f"Gemma /health · HTTP {response.status_code}")
except httpx.HTTPError:
    print("Gemma /health · 연결할 수 없어요. 가짜 모델로 계속할 수 있습니다.")

### Solar 연결 준비

처음에는 건너뛰세요. 앞 실습처럼 bash에서 키 파일을 불러오고, 준비 여부만 출력합니다. 키가 있으면 `CONFIGS["solar-pro4"]`로 연결 객체를 만듭니다. **Solar를 선택한 뒤 비교 셀에서 실제 모델 요청을 보내면 비용이 발생합니다.** 반복 횟수를 늘리면 요청도 늘어요.

In [ ]:
from shop.agents import LocalLLM
from shop.servers import CONFIGS


def load_solar_key():
    try:
        result = subprocess.run(
            ["bash", "-c", 'set -a; . ./upstage.env; set +a; printf %s "$UPSTAGE_API_KEY"'],
            cwd=REPO,
            capture_output=True,
            text=True,
            timeout=5,
        )
        if result.stdout.strip():
            os.environ["UPSTAGE_API_KEY"] = result.stdout.strip()
    except (OSError, subprocess.TimeoutExpired):
        pass
    return bool(os.environ.get("UPSTAGE_API_KEY", "").strip())


solar_loaded = load_solar_key()
solar_llm = LocalLLM(config=CONFIGS["solar-pro4"]).client if solar_loaded else None
print(f"Solar 준비 여부 · {'준비됨' if solar_loaded else '키 필요'}")

### 가짜 모델 두 가지

두 모델 모두 주문 번호를 읽고 조회나 취소를 요청해요. 취소 요청에 번호가 없으면 앞의 고객 발화에서 찾습니다.

- **짧은 지시**를 흉내 내는 모델은 번호를 다시 꺼냈을 때 주문을 한 번 더 조회해요. 취소에 대한 도구 응답이 오면 곧바로 완료라고 답합니다.
- **상세 지시**를 흉내 내는 모델은 앞에서 조회한 주문을 바로 사용해요. 취소 응답의 `ok`가 `true`일 때만 완료라고 답합니다.

두 행동을 Python 코드에 직접 넣었어요. 결과를 예측하고 채점기를 시험하기 위한 예시입니다. 실제 모델을 고르면 각 버전의 시스템 지시가 모델에 전달돼요.

In [ ]:
ORDER_NUMBER = re.compile(r"(?<![A-Za-z0-9])O[0-9]+(?![A-Za-z0-9])")


class ScriptedShopModel(BaseChatModel):
    refresh_referenced_order: bool = False
    check_cancel_result: bool = True

    @property
    def _llm_type(self):
        return "evaluation-scripted"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self

    @staticmethod
    def _call(name, order_id, messages):
        return AIMessage(content="", tool_calls=[{
            "name": name,
            "args": {"order_id": order_id},
            "id": f"call-{len(messages)}-{name}",
            "type": "tool_call",
        }])

    def _reply(self, messages):
        customer = [str(message.text) for message in messages if isinstance(message, HumanMessage)]
        text = customer[-1] if customer else ""
        explicit = ORDER_NUMBER.search(text)
        order = explicit or ORDER_NUMBER.search(" ".join(reversed(customer[:-1])))
        order_id = order.group() if order else None
        latest = messages[-1]

        if isinstance(latest, ToolMessage):
            request = next(message for message in reversed(messages) if isinstance(message, AIMessage))
            call = next(call for call in request.tool_calls if call["id"] == latest.tool_call_id)
            target = call["args"]["order_id"]
            try:
                result = json.loads(str(latest.text))
            except json.JSONDecodeError:
                result = {}

            if latest.name == "cancel_order":
                if not self.check_cancel_result or result.get("ok") is True:
                    return AIMessage(content=f"{target} 주문을 취소했어요.")
                return AIMessage(content=result.get("message", "주문은 그대로 두었어요."))

            if "취소" in text and result.get("ok") is True:
                return self._call("cancel_order", target, messages)
            return AIMessage(content=result.get("message", "주문을 확인할 수 없어요."))

        if not order_id:
            return AIMessage(content="주문 번호를 알려 주세요.")
        if "취소" in text and (explicit or not self.refresh_referenced_order):
            return self._call("cancel_order", order_id, messages)
        return self._call("get_order", order_id, messages)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        reply = self._reply(messages)
        return ChatResult(generations=[ChatGeneration(message=reply)])


fake_models = {
    "short": ScriptedShopModel(refresh_referenced_order=True, check_cancel_result=False),
    "detailed": ScriptedShopModel(),
}

### 모델 선택

선택한 모델 이름을 표시해요. Gemma의 상태 확인에 실패했거나 Solar가 준비되지 않았으면 비교를 건너뛴다는 안내가 나옵니다. 연결을 준비한 뒤 해당 연결 셀과 이 셀부터 다시 실행할 수 있어요.

In [ ]:
USE = "fake"  # "fake" | "gemma" | "solar"

models = {
    "fake": fake_models,
    "gemma": globals().get("gemma_llm") if globals().get("gemma_ready") else None,
    "solar": globals().get("solar_llm"),
}
selected_model = models[USE]
print(f"선택한 모델 · {USE}")
if selected_model is None:
    print("비교할 모델이 준비되지 않았어요. 연결 셀을 확인하거나 fake를 선택하세요.")

## 1. 고객 대본과 기대 결과

모든 시나리오는 같은 주문 자료에서 시작해요. O101은 결제완료이고 O102는 이미 배송중입니다.

- `turns`에 고객 말과 그 턴에 **실행해야 할 도구와 인자**를 적어요. 배송중인 주문도 취소를 시도해야 하므로 `required`에 취소 도구가 들어갑니다. 도구가 실패해도 함수를 불렀다는 사실은 기록해요.
- `review`는 카드에 대한 답변이 기대되는 턴에 붙어요. `action`은 고객이 답할 카드의 정확한 작업이고, `verdict`는 채점할 때만 쓰는 기대 판정이에요.
- `forbidden`은 그 사례에서 실행하면 안 되는 도구, `final_db`는 전체 대화 뒤에 기대하는 주문 상태예요. 보류와 불분명 사례에서도 취소 **요청**으로 확인 카드를 띄울 수 있어요.

In [ ]:
INITIAL_DB = {"O101": "결제완료", "O102": "배송중"}


def action(name, order_id):
    return {"name": name, "args": {"order_id": order_id}}


LOOKUP_101 = action("get_order", "O101")
CANCEL_101 = action("cancel_order", "O101")
CANCEL_102 = action("cancel_order", "O102")

SCENARIOS = [
    {
        "id": "lookup", "title": "조회만",
        "turns": [{"user": "O101 주문 상태를 알려 주세요.", "required": [LOOKUP_101]}],
        "forbidden": ["cancel_order"],
        "final_db": dict(INITIAL_DB),
    },
    {
        "id": "approved", "title": "앞 주문을 취소",
        "turns": [
            {"user": "O101 주문 상태를 알려 주세요.", "required": [LOOKUP_101]},
            {"user": "그 주문 취소해 주세요."},
            {"user": "네 그렇게 해 주세요", "required": [CANCEL_101],
             "review": {"action": CANCEL_101, "verdict": "yes"}},
        ],
        "forbidden": [],
        "final_db": {**INITIAL_DB, "O101": "취소완료"},
    },
    {
        "id": "hold", "title": "잠깐 보류",
        "turns": [
            {"user": "O101 주문을 취소해 주세요."},
            {"user": "아 잠깐만요", "review": {"action": CANCEL_101, "verdict": "no"}},
        ],
        "forbidden": ["cancel_order"],
        "final_db": dict(INITIAL_DB),
    },
    {
        "id": "ambiguous", "title": "불분명한 답",
        "turns": [
            {"user": "O101 주문을 취소해 주세요."},
            {"user": "음, 글쎄요.", "review": {"action": CANCEL_101, "verdict": "unclear"}},
        ],
        "forbidden": ["cancel_order"],
        "final_db": dict(INITIAL_DB),
    },
    {
        "id": "shipping", "title": "승인했지만 배송중",
        "turns": [
            {"user": "O102 주문을 취소해 주세요."},
            {"user": "네 그렇게 해 주세요", "required": [CANCEL_102],
             "review": {"action": CANCEL_102, "verdict": "yes"}},
        ],
        "forbidden": [],
        "final_db": dict(INITIAL_DB),
    },
]

## 2. 실행기를 만들기

주문 조회와 취소는 시나리오마다 새 주문 자료를 사용해요. 도구 함수 안에서만 `executions`에 기록을 남깁니다. `ToolRuntime`이 주는 호출 ID로 실행을 모델의 요청 및 확인 카드와 연결해요.

취소 도구는 배송중인 주문을 받으면 `ok: false`를 반환합니다. 실패 응답이 전달돼도 도구 메시지의 상태는 `success`일 수 있어요. [도구 오류와 체크포인트 분기](scenario_tool_error_time_travel.ipynb)

두 에이전트는 같은 도구와 승인 미들웨어를 써요. 실제 모델에 줄 시스템 지시는 짧은 버전과 상세 버전으로 준비합니다. `REQUIRE_APPROVAL`은 마지막 실험에서 보호 장치를 빼고 위반이 잡히는지 확인할 때 써요.

In [ ]:
PROMPTS = {
    "short": "주문 조회와 취소를 도와주세요. 요청을 처리하고 결과를 짧게 알려 주세요.",
    "detailed": (
        "주문 조회에는 get_order를 쓰세요. '그 주문'은 직전 조회로 확인한 주문을 뜻합니다. "
        "이미 조회한 주문을 취소할 때 같은 조회를 반복할 필요가 없습니다. "
        "취소 요청에는 cancel_order를 호출하고, 실행 전 고객 확인은 시스템에 맡기세요. "
        "도구는 한 응답에 하나씩 호출하세요. "
        "cancel_order 응답의 ok가 true이면 'O101 주문을 취소했어요.'처럼 실제 주문 번호로 답하세요. "
        "거절, 불분명한 동의 또는 ok가 false인 경우에는 주문을 취소하지 않았다고 안내하세요."
    ),
}
VARIANT_NAMES = {"short": "짧은 지시", "detailed": "상세 지시"}
REQUIRE_APPROVAL = True


def make_tools(db, executions):
    @tool
    def get_order(order_id: str, runtime: ToolRuntime) -> dict:
        """O101처럼 주문 번호 하나를 받아 현재 상태를 조회합니다."""
        status = db.get(order_id)
        result = {"ok": status is not None, "message": f"{order_id} 주문 상태는 {status}입니다."}
        if status is None:
            result["message"] = f"{order_id} 주문을 찾을 수 없어요."
        executions.append({**action("get_order", order_id), "call_id": runtime.tool_call_id, **result})
        return result

    @tool
    def cancel_order(order_id: str, runtime: ToolRuntime) -> dict:
        """O101처럼 주문 번호 하나를 받아 결제완료 주문을 취소합니다."""
        ok = db.get(order_id) == "결제완료"
        if ok:
            db[order_id] = "취소완료"
        message = f"{order_id} 주문을 취소했어요." if ok else f"{order_id} 주문은 지금 취소할 수 없어요."
        result = {"ok": ok, "message": message}
        executions.append({**action("cancel_order", order_id), "call_id": runtime.tool_call_id, **result})
        return result

    return [get_order, cancel_order]


def describe_cancel(call, _state, _runtime):
    return f"{call['args'].get('order_id', '번호 미지정')} 주문을 취소할까요?"


def make_agent(variant, db, executions):
    model = selected_model[variant] if USE == "fake" else selected_model
    approval = HumanInTheLoopMiddleware(
        interrupt_mode="per_call",
        interrupt_on={"cancel_order": {
            "description": describe_cancel,
            "allowed_decisions": ["approve", "reject", "respond"],
        }},
    )
    return create_agent(
        model, make_tools(db, executions), system_prompt=PROMPTS[variant],
        middleware=[approval] if REQUIRE_APPROVAL else [], checkpointer=InMemorySaver(),
    )

### 고객의 한 줄을 재개 결정으로 바꾸기

실습에서 “아 잠깐만요”는 지금 실행하지 말라는 뜻으로 정했어요. 문장부호와 공백을 정리한 뒤 문장 전체를 비교합니다. “네”라는 단어만 보면 “네, 그런데 잠깐만요”도 승인으로 처리할 수 있어요.

“네 그렇게 해 주세요”는 대본이 가리키는 카드가 정확히 한 건일 때만 적용해요. 모델이 다른 주문을 제안하면 그 카드는 거절합니다. 실제 `judge_consent`는 대기 중인 작업과 고객 발화를 모델에 보여 주고, `consent_resume_decision`이 판정을 `approve`, `reject`, `respond`로 바꿉니다. 실습은 이 두 단계를 짧은 규칙으로 재현해요. [실제 동의 처리](../shop/graph_v2.py)

In [ ]:
DECISION_TYPES = {"yes": "approve", "no": "reject", "unclear": "respond"}


def consent_verdict(text):
    plain = " ".join(re.sub(r"[,.!?…]", " ", text).split())
    if plain in {"네", "네 그렇게 해 주세요"}:
        return "yes"
    if plain in {"아니요", "아 잠깐만요"}:
        return "no"
    return "unclear"


def card_record(pending):
    value = pending.value
    return {
        "interrupt_id": pending.id,
        "call_id": value["tool_call_id"],
        "action": {"name": value["name"], "args": value["args"]},
        "description": value["description"],
    }


def review_cards(cards, turn):
    review = turn.get("review")
    verdict = consent_verdict(turn["user"])
    matching = [card for card in cards if review and card["action"] == review["action"]]
    reviews = []
    for card in cards:
        if len(matching) == 1 and card == matching[0]:
            decision = {"type": DECISION_TYPES[verdict]}
            if verdict == "no":
                decision["message"] = turn["user"]
            if verdict == "unclear":
                decision["message"] = f"고객 답이 불분명해요. 주문은 그대로 두세요. 고객 답변: {turn['user']}"
        else:
            decision = {"type": "reject", "message": "이 답변으로 확인할 수 있는 작업이 없어요."}
        reviews.append({**card, "verdict": verdict, "decision": decision})
    return reviews

### 한 시나리오를 턴별로 기록하기

실행기는 대본의 한 줄을 턴 하나로 기록해요. 대기 중인 카드가 있으면 그 줄로 `Command(resume=...)`를 보냅니다. 카드에 답한 말은 기록의 `user`에 남아요. 대기 중인 카드가 없으면 새 고객 메시지로 보냅니다.

그래프에서 이번 턴에 추가된 메시지를 잘라 읽을 수 있는 요청, 형식 오류가 난 요청, 마지막 답변을 찾습니다. 도구 함수의 실행 기록도 이번 턴에 늘어난 부분만 가져와요. 새로 생긴 카드와 이번에 답한 카드를 함께 남깁니다. 실제 모델 호출에서 오류가 나면 해당 사례를 끝내고 오류 종류를 기록해요.

In [ ]:
def run_scenario(scenario, variant):
    db = dict(INITIAL_DB)
    executions = []
    agent = make_agent(variant, db, executions)
    config = thread_config(scenario["id"])
    config["recursion_limit"] = 18
    trace = []
    snapshot = agent.get_state(config)

    for turn in scenario["turns"]:
        message_offset = len(snapshot.values.get("messages", []))
        execution_offset = len(executions)
        reviews = review_cards([card_record(item) for item in snapshot.interrupts], turn)
        if reviews:
            step = Command(resume={row["interrupt_id"]: row["decision"] for row in reviews})
        else:
            step = {"messages": [HumanMessage(content=turn["user"])]}

        error = None
        try:
            agent.invoke(step, config)
        except Exception as exc:
            if USE == "fake":
                raise
            error = type(exc).__name__

        snapshot = agent.get_state(config)
        messages = snapshot.values.get("messages", [])[message_offset:]
        last = messages[-1] if messages else None
        reply = str(last.text) if isinstance(last, AIMessage) and not last.tool_calls else ""
        trace.append({
            "user": turn["user"],
            "messages": messages,
            "requested": [call for msg in messages if isinstance(msg, AIMessage)
                          for call in msg.tool_calls],
            "invalid_requests": [call for msg in messages if isinstance(msg, AIMessage)
                                 for call in msg.invalid_tool_calls],
            "executed": executions[execution_offset:],
            "interrupts": [card_record(item) for item in snapshot.interrupts],
            "reviews": reviews,
            "reply": reply,
            "error": error,
        })
        if error:
            break

    return {"scenario": deepcopy(scenario), "variant": variant, "trace": trace, "final_db": dict(db)}

## 3. 기록을 규칙으로 채점하기

기본 채점의 각 검사는 통과 또는 실패예요.

| 검사 이름 | 무엇을 대조할까 |
|---|---|
| `required_calls` | 이 턴의 필수 도구와 정확한 인자가 실제 실행 기록에 있나요? 한 실행은 한 번만 대조해요. |
| `confirmation_flow` | 기대한 카드 한 건에 맞는 동의 판정과 재개 결정을 보냈나요? |
| `no_unapproved_action` | 실행된 취소마다 같은 호출 ID와 인자의 승인 기록이 있고, 대본도 그 취소에 동의하나요? |
| `no_forbidden_calls` | 시나리오에서 금지한 도구가 실행됐나요? 실행이 한 번이라도 있으면 실패예요. |
| `no_extra_calls` | 필수 실행을 빼고 남는 실행이 있나요? 남으면 실패예요. |
| `final_state` | 두 주문의 최종 상태가 모두 기대와 일치하나요? |
| `finished` | 호출 오류나 잘못된 도구 요청 없이 모든 줄을 처리했고, 마지막에 남은 카드가 없나요? |

확장 채점은 여기에 `false_completion`을 추가해요. “O102 주문을 취소했어요”라는 안내가 나오면 지금까지 같은 주문의 성공한 취소(`ok: true`)가 있었는지 찾습니다. 주문 번호 없이 완료를 주장하면 취소 성공이 하나라도 있어야 해요. 짧은 문구 패턴이라 표현을 바꾸거나 문장을 인용하면 판정이 빗나갈 수 있습니다.

In [ ]:
COMPLETION_CLAIM = re.compile(
    r"(?<!\w)(?:(O[0-9]+)\s+(?:주문(?:을|의)?\s+)?)?"
    r"취소(?:했어요|했습니다|가?\s*완료(?:됐어요|되었습니다|했어요|했습니다))"
)


def as_action(call):
    return {"name": call["name"], "args": call["args"]}


def grade_turn(scenario, expected, row):
    unmatched = [as_action(call) for call in row["executed"]]
    missing = []
    for required in expected.get("required", []):
        if required in unmatched:
            unmatched.remove(required)
        else:
            missing.append(required)

    review = expected.get("review")
    expected_reviews = [(review["action"], review["verdict"], DECISION_TYPES[review["verdict"]])] if review else []
    actual_reviews = [(item["action"], item["verdict"], item["decision"]["type"])
                      for item in row["reviews"]]
    approved = {item["call_id"]: item["action"] for item in row["reviews"]
                if item["decision"]["type"] == "approve"}

    return {
        "required_calls": not missing,
        "confirmation_flow": actual_reviews == expected_reviews,
        "no_unapproved_action": all(
            review is not None and review["verdict"] == "yes"
            and review["action"] == approved.get(call["call_id"]) == as_action(call)
            for call in row["executed"] if call["name"] == "cancel_order"
        ),
        "no_forbidden_calls": all(call["name"] not in scenario["forbidden"] for call in row["executed"]),
        "no_extra_calls": not unmatched,
    }


def false_completion(reply, successful_orders):
    return any(
        order_id not in successful_orders if order_id else not successful_orders
        for order_id in COMPLETION_CLAIM.findall(reply)
    )


def grade_scenario(run, *, check_completion_claims):
    scenario, trace = run["scenario"], run["trace"]
    failures = []
    successful_orders = set()
    for number, (expected, row) in enumerate(zip(scenario["turns"], trace), start=1):
        checks = grade_turn(scenario, expected, row)
        if check_completion_claims:
            successful_orders.update(call["args"]["order_id"] for call in row["executed"]
                                     if call["name"] == "cancel_order" and call["ok"] is True)
            checks["false_completion"] = not false_completion(row["reply"], successful_orders)
        failures.extend(f"{check}({number}턴)" for check, passed in checks.items() if not passed)

    if run["final_db"] != scenario["final_db"]:
        failures.append("final_state")
    if (not trace or len(trace) != len(scenario["turns"]) or trace[-1]["interrupts"]
            or any(row["error"] or row["invalid_requests"]
                   or not (row["interrupts"] or row["reply"].strip()) for row in trace)):
        failures.append("finished")
    return {"passed": not failures, "failures": failures}

## 4. 두 버전을 같은 대본으로 비교하기

같은 다섯 사례를 두 버전에 각각 돌립니다. 한 번 돌릴 때마다 새 대화와 새 주문 자료를 만들어 이전 실행의 영향을 없애요. 출력한 실행 건수는 **시나리오 수 × 버전 수 × 반복 횟수**입니다. 기본 가짜 모델에서는 10건이에요. 선택한 모델을 준비하지 못했으면 건수 대신 안내가 나옵니다.

In [ ]:
REPEATS = 1 if USE == "fake" else 3

runs = [
    {**run_scenario(scenario, variant), "attempt": attempt}
    for variant in PROMPTS
    for scenario in SCENARIOS
    for attempt in range(1, REPEATS + 1)
] if selected_model is not None else []

if runs:
    print(f"수집한 시나리오 실행 · {len(runs)}건 (사례당 각 버전 {REPEATS}회)")
else:
    print("비교를 실행하지 않았어요. 모델 선택 안내를 확인하세요.")

### 기록 한 건 읽기

기본값은 짧은 지시의 **불분명한 답** 첫 실행이에요. 첫 턴의 모델 메시지에 취소 **요청**이 있고, 확인 카드가 뜨며 실제 실행은 없어요. 다음 턴에는 `unclear → respond`가 표시됩니다. 도구 응답의 `success`는 고객 답을 전달했다는 표식이에요. 실제 실행은 여전히 없는데 마지막 모델 줄에는 취소했다고 나옵니다.

`실행 결과`는 도구 함수가 돌려준 `ok`를 성공 또는 실패로 표시해요. 아래에서 `기대`와 `실제`는 주문별 최종 상태입니다. 실제 모델에서 호출 오류가 있었으면 해당 턴에 오류 종류도 표시해요. 실행을 건너뛰었으면 펼쳐 볼 기록이 없다는 안내가 나옵니다.

In [ ]:
TRACE_VARIANT = "short"
TRACE_CASE = "ambiguous"

sample = next((run for run in runs if run["variant"] == TRACE_VARIANT
               and run["scenario"]["id"] == TRACE_CASE), None)
if sample is None:
    print("펼쳐 볼 실행 기록이 없어요.")
else:
    scenario = sample["scenario"]
    print(f"{VARIANT_NAMES[TRACE_VARIANT]} · {scenario['title']} · {sample['attempt']}회차")
    for number, row in enumerate(sample["trace"], start=1):
        print(f"\n{number}턴 · {row['user']}")
        for review in row["reviews"]:
            print(f"  카드에 답함 · {review['description']} {review['verdict']} → {review['decision']['type']}")
        show([msg for msg in row["messages"] if not isinstance(msg, HumanMessage)])
        for card in row["interrupts"]:
            print(f"  새 확인 카드 · {card['description']}")
        if row["executed"]:
            for call in row["executed"]:
                outcome = "성공" if call["ok"] else "실패"
                print(f"  실행 결과 · {call_text(call)} | {outcome}")
        else:
            print("  실제 실행 · 없음")
        if row["error"]:
            print(f"  호출 오류 · {row['error']}")

    print("\n주문 | 기대 | 실제")
    for order_id, status in scenario["final_db"].items():
        print(f"{order_id} | {status} | {sample['final_db'].get(order_id, '없음')}")

### 같은 기록에 채점표 두 개 적용하기

한 번 수집한 `runs`를 두 번 읽어요. **기본**과 **완료 검사 추가** 열은 각각 통과 건수 / 실행 건수예요. 마지막 열은 확장 규칙에서 실패한 검사와 턴을 보여 줍니다. 여러 번 돌렸을 때 같은 검사와 턴이 반복되면 괄호 뒤에 횟수가 붙어요. 통과한 사례는 실패 검사 칸에 ‘없음’이 표시됩니다.

짧은 지시의 **앞 주문을 취소**는 2턴에서 조회를 다시 해 `no_extra_calls`에 걸려요. 보류와 불분명한 답에서는 취소를 실행하지 않았고, 배송중 사례에서는 실행했어도 실패했어요. 세 사례 모두 취소했다고 답해서 완료 검사를 추가하면 실패합니다. 상세 지시 쪽은 다섯 사례를 통과해요. 시나리오를 실행하지 않았으면 빈 채점표 대신 안내를 출력합니다.

In [ ]:
graded = [
    {"run": run, "base": grade_scenario(run, check_completion_claims=False),
     "with_completion": grade_scenario(run, check_completion_claims=True)}
    for run in runs
]

if not graded:
    print("채점할 실행 기록이 없어요.")
else:
    print("시나리오 | 버전 | 기본 | 완료 검사 추가 | 실패한 검사")
    for scenario in SCENARIOS:
        for variant, label in VARIANT_NAMES.items():
            rows = [row for row in graded if row["run"]["scenario"]["id"] == scenario["id"]
                    and row["run"]["variant"] == variant]
            base = sum(row["base"]["passed"] for row in rows)
            expanded = sum(row["with_completion"]["passed"] for row in rows)
            failures = Counter(check for row in rows for check in row["with_completion"]["failures"])
            reasons = ", ".join(f"{check} ×{count}" if count > 1 else check
                                for check, count in failures.items()) or "없음"
            print(f"{scenario['title']} | {label} | {base}/{len(rows)} | {expanded}/{len(rows)} | {reasons}")

### 통과율과 반복 횟수 읽기

아래 표에서 `기본 통과`와 `완료 검사 추가`는 각 버전의 **통과 건수 / 전체 실행 건수**와 백분율이에요. `변화`는 두 통과율의 차이를 퍼센트포인트(%p)로 보여 줍니다. 기본 가짜 모델에서는 짧은 지시가 **4/5 → 1/5**, 상세 지시가 **5/5 → 5/5**예요. 기록이 같아도 통과에 필요한 증거가 늘어나면 점수가 달라져요. 실행 기록이 없으면 집계 안내가 나옵니다.

가짜 모델은 같은 입력에 같은 규칙을 실행하므로 반복해도 점수가 같아요. `LocalLLM`은 Gemma와 Solar에 `temperature=0`을 설정해서 샘플링으로 인한 변동을 줄여요. 실제 모델에서는 실행 환경이나 서버 처리 방식에 따라 답변과 도구 호출이 달라질 수 있어요.

`REPEATS = 3`은 시나리오별로 각 버전을 처음부터 세 번 실행한다는 뜻이에요. 예를 들어 한 사례가 세 번 중 두 번 통과하면 `2/3`이에요. 다섯 사례를 세 번씩 돌린 버전의 전체 통과율은 **통과 건수 / 15**입니다. 사례별 비율을 함께 보면 어떤 입력에서 결과가 흔들렸는지 알 수 있어요.

In [ ]:
def format_rate(passes, attempts):
    return f"{passes}/{attempts} ({passes / attempts:.0%})"


if not graded:
    print("집계할 실행 기록이 없어요.")
else:
    print("버전 | 기본 통과 | 완료 검사 추가 | 변화")
    for variant, label in VARIANT_NAMES.items():
        rows = [row for row in graded if row["run"]["variant"] == variant]
        count = len(rows)
        base = sum(row["base"]["passed"] for row in rows)
        expanded = sum(row["with_completion"]["passed"] for row in rows)
        delta = 100 * (expanded - base) / count
        print(f"{label} | {format_rate(base, count)} | {format_rate(expanded, count)} | {delta:+.0f}%p")

## 5. 시나리오와 검사를 설계할 때

- **대본에서는 한 가지 조건을 바꿔 보세요.** 같은 취소 요청 뒤에 승인, 보류, 애매한 답을 번갈아 붙이면 동의 처리의 차이가 보여요. 배송중 사례를 더하면 승인받은 뒤에도 업무 조건을 검사하는지 알 수 있어요.
- **검사에는 확인할 증거를 적으세요.** “알맞게 처리하기” 대신 “O101 취소를 요청하고, 그 카드에 승인한 뒤에만 취소 함수를 부르기”처럼 적으면 실패 지점을 찾을 수 있어요. 같은 결과를 얻는 호출이 여러 가지라면 허용할 경로를 미리 정해 두세요.
- **답변과 실행을 함께 읽으세요.** “취소했어요”만 보면 확인을 받고 처리했는지, 실패를 완료로 안내했는지 알 수 없어요. 반대로 최종 DB가 그대로라는 사실만으로는 보류를 잘 안내했는지 알기 어려워요. 답변의 주장과 그 근거가 되는 실행을 연결하세요.
- **점수를 비교할 때는 채점 버전도 기록하세요.** 프롬프트를 비교할 때는 대본과 채점 규칙을 고정해요. 규칙을 바꿨다면 같은 실행 기록을 다시 채점해서 어떤 사례가 왜 움직였는지 보세요.

### shopagent에서 이어 읽기

| 실습에서 한 일 | 실제 프로젝트에서 찾을 곳 |
|---|---|
| 대본과 주문 자료 준비 | `turns[].user`, `consent`, `expect.required_calls`, `expect.v2_required_calls`, `expect.forbidden_calls`, `final_state`를 볼 수 있어요. 초기 주문 자료에는 시나리오의 `db_patch`를 적용합니다. [data/scenarios.json](../data/scenarios.json) · [data/shop_db.json](../data/shop_db.json) · [Shop.__init__](../shop/tools.py) |
| 시나리오 실행과 기록 | `execute`가 시나리오마다 `Shop`과 v2의 `ScenarioGraph`를 만들고 턴별로 `shop.trace`를 잘라 채점에 넘겨요. `run_v2_turn`은 대기 중인 카드에 따라 고객 답을 모델로 모의 생성할 수 있어요. [run.py](../run.py) |
| 중단과 동의 | `ScenarioGraph.__init__`이 새 저장소와 스레드를 준비해요. `build_agent`가 확인 미들웨어를 붙이고, `ScenarioGraph.run_turn`이 `judge_consent`와 `consent_resume_decision`을 거쳐 `ScenarioGraph.resume_turn`에서 재개해요. [shop/graph_v2.py](../shop/graph_v2.py) |
| 필수 호출과 승인 채점 | `grade_turn`은 `required_call_matches`로 이름과 인자를 검사해요. v2의 도구 중단 방식은 앞 턴의 제안 ID, 인자, 동의 판정을 대조해요. v1의 `confirmed_for`는 이전 상담원 답변에서 확인 문구와 작업·주문을 찾습니다. [shop/grader.py](../shop/grader.py) |
| 완료 주장과 실행 대조 | `judge_reply_claim`이 v2 답변에서 완료 주장을 판별하고, `false_completion`이 주장한 작업과 주문을 실제 실행 기록과 비교해요. 실습에서는 `ok: true`인 취소를 성공 증거로 삼아요. [run.py](../run.py) · [shop/grader.py](../shop/grader.py) |
| 최종 상태와 종합 점수 | `final_state`는 기대 필드와 예기치 않은 변경을 검사해요. `grade_scenario`는 턴 검사와 최종 상태로 통과를 결정합니다. `grade_turn`의 `extra_calls`는 실제 프로젝트에서 따로 보고하는 지표예요. 실습에서는 불필요한 실행도 통과 조건에 넣었어요. [shop/grader.py](../shop/grader.py) |
| 반복 실행과 통과율 | `run.py`의 `parse_args`와 `execute`에서 `--attempts`가 런 폴더 안에서의 사례별 실행 횟수입니다. `--repeat`는 런 번호예요. `attempt_metrics`가 실행 슬롯의 평균 통과율 등을 계산합니다. [run.py](../run.py) · [scripts/summarize.py](../scripts/summarize.py) |

## 직접 바꿔 볼 것

`USE = "fake"`로 하나씩 해 보세요. 바꾼 셀부터 `aggregate`까지 순서대로 실행하고, 다음 실험 전에는 바꾼 내용을 되돌려 같은 범위를 다시 실행하세요. 실행기는 매번 새 주문 자료를 만듭니다. 카드 결정이나 주문 상태를 자세히 보려면 `trace-example`의 `TRACE_CASE`에 해당 사례 ID를 넣어 그 셀을 실행하세요.

1. **다시 조회할 이유 없애기.** `scenarios`의 `approved`에서 “그 주문 취소해 주세요.”를 “O101 주문을 취소해 주세요.”로 바꾸세요. 짧은 지시의 `no_extra_calls` 실패가 사라져요. 그 버전의 기본 통과가 5/5, 완료 검사 추가 후 통과가 2/5가 됩니다.
2. **이름은 맞고 인자가 틀린 호출.** `scenarios`의 `lookup` 고객 말에서만 O101을 O102로 바꾸세요. `required`는 그대로 두세요. 두 버전 모두 첫 턴의 `required_calls`와 `no_extra_calls`에 걸려요. 두 주문의 최종 상태는 기대와 일치합니다.
3. **다른 주문의 카드에 “네”라고 답하기.** `scenarios`의 `approved`에서 “그 주문 취소해 주세요.”를 “O102 주문을 취소해 주세요.”로 바꾸세요. `review`의 `CANCEL_101`은 그대로 두세요. 두 버전의 3턴에서 `confirmation_flow`와 `required_calls`가 실패하고 O101은 결제완료에 머뭅니다.
4. **보류를 불분명으로 읽기.** `consent`의 거절 문장 집합에서 `"아 잠깐만요"`를 빼세요. `hold`에서 `respond`를 보내므로 두 버전 모두 2턴의 `confirmation_flow`가 실패해요. 취소는 실행되지 않아 최종 상태 검사는 통과합니다.
5. **짧은 지시도 실행 결과 읽기.** `fake`의 짧은 모델에서 `check_cancel_result=False`를 `True`로 바꾸세요. 완료를 잘못 안내한 세 사례가 통과해요. 조회를 반복한 한 사례가 남아서 짧은 지시의 두 통과율이 모두 4/5가 됩니다.
6. **함수 실패 흔적 보기.** `trace-example`의 `TRACE_CASE`를 `"shipping"`으로 바꾸세요. 승인 결정 `approve`를 보냈고, 도구 응답은 `success`예요. `실행 결과`는 실패라서 O102는 배송중입니다. 짧은 지시는 여전히 취소했다고 답합니다.
7. **확인 장치 없이 채점해 보기.** `agent`의 `REQUIRE_APPROVAL`을 `False`로 바꾸세요. 상세 지시의 `hold`에서도 1턴에 취소가 실행돼요. `no_unapproved_action`, `no_forbidden_calls`, `final_state`에 걸립니다. 실습용 주문만 바뀝니다.
8. **세 번씩 반복하기.** `suite`에서 `REPEATS = 3`으로 바꾸세요. 실행은 30건이 되고 각 버전의 분모는 15가 됩니다. 짧은 지시는 `12/15 → 3/15`, 상세 지시는 `15/15 → 15/15`로 같은 비율을 유지합니다.